In [ ]:
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns
import tensorflow as tf
from tensorflow import keras
#from keras.optimizers import Adam
from keras.losses import Loss
from keras.initializers import HeNormal, HeUniform, GlorotUniform
from keras.utils import to_categorical
from joblib import Parallel, delayed, parallel_backend
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

## GPU visibility

In [ ]:
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
#print(tf.config.experimental.list_physical_devices())
gpus = tf.config.list_physical_devices('GPU'); print(gpus)
tf.config.set_visible_devices([gpus[0]], 'GPU')
tf.config.get_visible_devices('GPU')

In [ ]:
# Avoid memory growth
gpus = tf.config.get_visible_devices('GPU') # tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            #tf.config.experimental.set_virtual_device_configuration(gpu, [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=3774)])
            print(f"Enabled memory growth for GPU: {gpu.name}")
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs configured.")
    except RuntimeError as e:
        print(f"RuntimeError during GPU configuration: {e}")
        print("Ensure GPU memory configuration is set at the very beginning of your script.")
else:
    print("No GPU devices found. Running on CPU.")

# CIFAR-10 Data

In [ ]:
# Load the MNIST dataset
(X1, y1), (X2, y2) = keras.datasets.cifar10.load_data()
X = np.concatenate((X1, X2), axis=0)
X = X / 255.0 # Normalize the pixel values to [0, 1]
Y = np.concatenate((y1, y2), axis=0)[:,0]
print("Images shape:", X.shape)
print("Labels shape:", Y.shape)

In [ ]:
# Contamination introducer
def corrupt_labels(y_train, prob, num_classes=10, seed=None):
    if seed is not None:
        np.random.seed(seed)

    y_corrupted = y_train.copy()
    n = len(y_train)
    # Decide which labels to corrupt
    corrupt_mask = np.random.rand(n) < prob
    # For each label to corrupt, choose a new label different from the original
    for i in np.where(corrupt_mask)[0]:
        original_label = y_train[i]
        # possible new labels excluding the original
        new_labels = list(range(num_classes))
        new_labels.remove(original_label)
        # randomly pick a new label
        y_corrupted[i] = np.random.choice(new_labels)

    return y_corrupted

In [ ]:
## Loss functions

class TSCCE(Loss):
    def __init__(self, trim_ratio=0.2):
        super().__init__()
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        # Clip predictions to avoid log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        log_probs = tf.math.log(y_pred)
        # Get log probability of the correct class for each sample
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        per_sample_loss = -tf.gather_nd(log_probs, indices)
        # Trim top X% highest-loss samples
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_values, _ = tf.math.top_k(-per_sample_loss, k=k, sorted=False)
        trimmed_loss = -tf.reduce_mean(trimmed_values)

        return trimmed_loss
    
class TDPDSCCE(Loss):
    def __init__(self, alpha, trim_ratio):
        super().__init__()
        self.alpha = float(alpha)
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  tf.reduce_sum(y_pred**(self.alpha+1), axis=1) - (1+1/self.alpha)*(sel_probs**self.alpha)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals
        
        return tf.reduce_mean(trimmed_losses)

class LDPD(Loss):
    def __init__(self, alpha, trim_ratio):
        super().__init__()
        self.alpha = float(alpha)
        self.trim_ratio = float(trim_ratio)        

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses = tf.math.log(tf.reduce_sum(y_pred**(self.alpha+1), axis=1)) - (self.alpha+1)*tf.math.log(sel_probs)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals
        
        return tf.reduce_mean(trimmed_losses)

class SDIV(Loss):
    def __init__(self, alpha, lam, trim_ratio):
        super().__init__()
        self.alpha = float(alpha)
        self.lam = float(lam)
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        A = 1 + self.lam*(1 - self.alpha)
        B = self.alpha - self.lam*(1 - self.alpha)
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  (tf.reduce_sum(y_pred**(self.alpha+1), axis=1))/A - ((1+self.alpha)/(A*B))*(sel_probs**B)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals
        
        return tf.reduce_mean(trimmed_losses)

# Model

In [ ]:
def get_model():
    model = keras.Sequential([
        keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', kernel_initializer=HeNormal(seed=42)),
        keras.layers.MaxPooling2D((2, 2)),
        
        keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', kernel_initializer=HeNormal(seed=42)),
        keras.layers.MaxPooling2D((2, 2)),
        
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation='relu', kernel_initializer=HeNormal(seed=42)),
        keras.layers.Dense(10, activation='softmax', kernel_initializer=GlorotUniform(seed=42))     
    ])
    return model

### Fold id

In [ ]:
fold_id = 1
train_idx = pd.read_csv('train_indices.csv', header=None).iloc[:,fold_id]
val_idx = pd.read_csv('test_indices.csv', header=None).iloc[:,fold_id]
len(train_idx), len(val_idx)

In [ ]:
n_epochs = 250
def ACC_CCE(delta, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    model.fit(X_train, y_train_corrupted, epochs = n_epochs, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

def ACC_TSCCE(delta, trim_ratio, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=TSCCE(trim_ratio=trim_ratio))
    model.fit(X_train, y_train_corrupted, epochs = n_epochs, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

def ACC_TDPD(delta, alpha, trim_ratio, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=TDPDSCCE(alpha=alpha, trim_ratio=trim_ratio))
    model.fit(X_train, y_train_corrupted, epochs = n_epochs, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

def ACC_LDPD(delta, alpha, trim_ratio, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=LDPD(alpha=alpha, trim_ratio=trim_ratio))
    model.fit(X_train, y_train_corrupted, epochs = n_epochs, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

def ACC_SDIV(delta, alpha, lam, trim_ratio, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=SDIV(alpha=alpha, lam=lam, trim_ratio=trim_ratio))
    model.fit(X_train, y_train_corrupted, epochs = n_epochs, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

def ACC_MAE(delta, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    y_train_corrupted = to_categorical(y_train_corrupted, 10)
    model.compile(optimizer='adam', loss='mae')
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

# CCE

In [ ]:
dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
nj = len(dl)

def inner_acc_cce(X_train, y_train, X_test, y_test):
    row_result = []
    for delta in dl:
        result = ACC_CCE(delta, X_train, y_train, X_test, y_test)
        row_result.append(result)
    return np.array(row_result).reshape(1, 2 * nj)

def ACC_CCE_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_cce(X_train, y_train, X_test, y_test)

In [ ]:
results = ACC_CCE_fold(train_idx, val_idx)
df = pd.DataFrame(results, columns = np.repeat(dl,2), index = ['CCE'])
df.to_csv('CCE-F1.csv')
df

# TSCCE

In [ ]:
dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
t_ratios = [0.1,0.2,0.3]

nj = len(dl); l_tr = len(t_ratios)
def inner_acc_tscce(X_train, y_train, X_test, y_test):
    results = []
    for tr in t_ratios:
        row_result = []
        for delta in dl:
            result = ACC_TSCCE(delta, tr, X_train, y_train, X_test, y_test)
            row_result.append(result)
        results.append(row_result)
    return np.array(results).reshape(l_tr, 2 * nj)

def ACC_TSCCE_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_tscce(X_train, y_train, X_test, y_test)

In [ ]:
results = ACC_TSCCE_fold(train_idx, val_idx)
tscce_cv = pd.DataFrame(results, columns = np.repeat(dl,2), index = t_ratios)
tscce_cv.to_csv('TSCCE-F1.csv')
tscce_cv

# S-Divergence

In [ ]:
dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
#al = np.array([0.1,0.3,0.5,0.7,1.0])
al = np.array([0.3])

nj = len(dl); l_al = len(al)

def inner_acc_sdiv(X_train, y_train, X_test, y_test):
    trim_ratio = 0.0
    lam = -0.7
    results = []
    for alpha in al:
        row_result = []
        for delta in dl:
            result = ACC_SDIV(delta, alpha, lam, trim_ratio, X_train, y_train, X_test, y_test)
            row_result.append(result)
        results.append(row_result)
    return np.array(results).reshape(l_al, 2 * nj)


def ACC_SDIV_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_sdiv(X_train, y_train, X_test, y_test)

In [ ]:
results = ACC_SDIV_fold(train_idx, val_idx)
Sdiv_cv = pd.DataFrame(results, columns = np.repeat(dl,2), index = al)
Sdiv_cv.to_csv('SDIV-ldm0.7-al0.3-F1.csv')
Sdiv_cv

In [ ]:
dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
al = np.array([0.0, 0.5, 0.7, 1.0])

nj = len(dl); l_al = len(al)

def inner_acc_sdiv(X_train, y_train, X_test, y_test):
    trim_ratio = 0.0
    lam = -0.7
    results = []
    for alpha in al:
        row_result = []
        for delta in dl:
            result = ACC_SDIV(delta, alpha, lam, trim_ratio, X_train, y_train, X_test, y_test)
            row_result.append(result)
        results.append(row_result)
    return np.array(results).reshape(l_al, 2 * nj)


def ACC_SDIV_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_sdiv(X_train, y_train, X_test, y_test)

results = ACC_SDIV_fold(train_idx, val_idx)
Sdiv_cv = pd.DataFrame(results, columns = np.repeat(dl,2), index = al)
Sdiv_cv.to_csv('SDIV-ldm0.7-al51-F1.csv')
Sdiv_cv

In [ ]:
## $\lambda = -0.5$

## $\lambda = -0.5$

In [ ]:
dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
al = np.array([0.0, 0.1, 0.3])

nj = len(dl); l_al = len(al)
def inner_acc_sdiv(X_train, y_train, X_test, y_test):
    trim_ratio = 0.0
    lam = -0.5
    results = []
    for alpha in al:
        row_result = []
        for delta in dl:
            result = ACC_SDIV(delta, alpha, lam, trim_ratio, X_train, y_train, X_test, y_test)
            row_result.append(result)
        results.append(row_result)
    return np.array(results).reshape(l_al, 2 * nj)

def ACC_SDIV_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_sdiv(X_train, y_train, X_test, y_test)

results = ACC_SDIV_fold(train_idx, val_idx)
Sdiv_cv = pd.DataFrame(results, columns = np.repeat(dl,2), index = al)
Sdiv_cv.to_csv('SDIV-ldm0.5-al03-F1.csv')
print(Sdiv_cv)

###################################################
###################################################

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
al = np.array([0.5, 0.7, 1.0])

nj = len(dl); l_al = len(al)
def inner_acc_sdiv(X_train, y_train, X_test, y_test):
    trim_ratio = 0.0
    lam = -0.5
    results = []
    for alpha in al:
        row_result = []
        for delta in dl:
            result = ACC_SDIV(delta, alpha, lam, trim_ratio, X_train, y_train, X_test, y_test)
            row_result.append(result)
        results.append(row_result)
    return np.array(results).reshape(l_al, 2 * nj)

def ACC_SDIV_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_sdiv(X_train, y_train, X_test, y_test)

results = ACC_SDIV_fold(train_idx, val_idx)
Sdiv_cv = pd.DataFrame(results, columns = np.repeat(dl,2), index = al)
Sdiv_cv.to_csv('SDIV-ldm0.5-al51-F1.csv')
Sdiv_cv

## $\lambda = -1, 0.5$

In [ ]:
dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
al = np.array([0.5, 0.7])

nj = len(dl); l_al = len(al)
def inner_acc_sdiv(X_train, y_train, X_test, y_test):
    trim_ratio = 0.0
    lam = -1.0
    results = []
    for alpha in al:
        row_result = []
        for delta in dl:
            result = ACC_SDIV(delta, alpha, lam, trim_ratio, X_train, y_train, X_test, y_test)
            row_result.append(result)
        results.append(row_result)
    return np.array(results).reshape(l_al, 2 * nj)

def ACC_SDIV_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_sdiv(X_train, y_train, X_test, y_test)

results = ACC_SDIV_fold(train_idx, val_idx)
Sdiv_cv = pd.DataFrame(results, columns = np.repeat(dl,2), index = al)
Sdiv_cv.to_csv('SDIV-ldm1-al57-F1.csv')
print(Sdiv_cv)

###################################################
###################################################

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
al = np.array([0.5, 0.7])

nj = len(dl); l_al = len(al)
def inner_acc_sdiv(X_train, y_train, X_test, y_test):
    trim_ratio = 0.0
    lam = 0.5
    results = []
    for alpha in al:
        row_result = []
        for delta in dl:
            result = ACC_SDIV(delta, alpha, lam, trim_ratio, X_train, y_train, X_test, y_test)
            row_result.append(result)
        results.append(row_result)
    return np.array(results).reshape(l_al, 2 * nj)

def ACC_SDIV_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_sdiv(X_train, y_train, X_test, y_test)

results = ACC_SDIV_fold(train_idx, val_idx)
Sdiv_cv = pd.DataFrame(results, columns = np.repeat(dl,2), index = al)
Sdiv_cv.to_csv('SDIV-ld0.5-al57-F1.csv')
Sdiv_cv

# MAE

In [ ]:
dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
nj = len(dl)

def inner_acc_mae(X_train, y_train, X_test, y_test):
    row_result = []
    for delta in dl:
        result = ACC_MAE(delta, X_train, y_train, X_test, y_test)
        row_result.append(result)
    return np.array(row_result).reshape(1, 2 * nj)

def ACC_MAE_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_mae(X_train, y_train, X_test, y_test)

results = ACC_MAE_fold(train_idx, val_idx)
df = pd.DataFrame(results, columns = np.repeat(dl,2), index = ['MAE'])
df.to_csv('MAE-F1.csv')
df